In [1]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import lightning as L
from lightning.pytorch import seed_everything
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import MLFlowLogger

from spectral_core import compute_metrics, plot_predictions
from spectral_core.cnn1d.dataset import SpectraDataModule
from spectral_core.cnn1d.models import Lightning1DCNNModel

/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [18]:
DATA_DIR = Path('../data/processed/')
MODEL_DIR = Path('../data/models/')
FIGURE_DIR = Path('../data/figures/')
TRACKING_URI = f"sqlite:///{Path('../data/mlflow.db').resolve()}"
CHECKPOINT_DIR = MODEL_DIR / '_checkpoints'
RESULTS_PATH = MODEL_DIR / 'cnn_seeds.json'

INPUT_SIZE = 3319
LEARNING_RATE = 1e-4
BATCH_SIZE = 32
MAX_EPOCHS = 300
EARLY_STOPPING_PATIENCE = 15
LOG_EVERY_N_STEPS = 13

REPORTED_SEED = 0
SEEDS = tuple(range(10))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
reference = Lightning1DCNNModel(input_size=INPUT_SIZE, learning_rate=LEARNING_RATE)

total = sum(p.numel() for p in reference.parameters())
trainable = sum(p.numel() for p in reference.parameters() if p.requires_grad)
n_train = len(np.load(DATA_DIR / 'y_train.npy'))

print(f'total parameters      {total:,}')
print(f'trainable parameters  {trainable:,}')
print(f'training samples      {n_train}')
print(f'parameters per sample {total / n_train:,.0f}')
print()
for name, parameter in reference.named_parameters():
    if parameter.numel() > 10_000:
        print(f'  {name:<28}{parameter.numel():>14,}')

total parameters      19,302,065
trainable parameters  19,302,065
training samples      403
parameters per sample 47,896

  model.conv1d2.weight                20,480
  model.fc.fc1.weight             19,215,360
  model.fc.fc2.weight                 64,800


In [21]:
def train_one(seed):
    seed_everything(seed, workers=True, verbose=False)

    datamodule = SpectraDataModule(data_folder=str(DATA_DIR), batch_size=BATCH_SIZE)
    model = Lightning1DCNNModel(input_size=INPUT_SIZE, learning_rate=LEARNING_RATE)

    checkpoint = ModelCheckpoint(
        monitor='val_loss',
        dirpath=CHECKPOINT_DIR / f'seed{seed}',
        filename='best',
        save_top_k=1,
        mode='min'
    )
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOPPING_PATIENCE,
        mode='min'
    )
    logger = MLFlowLogger(
        experiment_name='1D_CNN_Regression',
        run_name=f'cnn_seed{seed}',
        tracking_uri=TRACKING_URI
    )
    logger.log_hyperparams({
        'seed': seed,
        'input_size': INPUT_SIZE,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE
    })

    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='cpu',
        logger=logger,
        callbacks=[checkpoint, early_stopping],
        log_every_n_steps=LOG_EVERY_N_STEPS,
        enable_progress_bar=False,
        enable_model_summary=False
    )
    trainer.fit(model, datamodule)

    best = Lightning1DCNNModel.load_from_checkpoint(checkpoint.best_model_path)
    best.eval()

    X_test = np.load(DATA_DIR / 'X_test.npy').astype(np.float32)
    y_test = np.load(DATA_DIR / 'y_test.npy').astype(np.float32)
    with torch.no_grad():
        y_pred = best(torch.tensor(X_test)).squeeze(-1).numpy()

    metrics = compute_metrics(y_test, y_pred)
    logger.log_metrics({f'test_{name}': value for name, value in metrics.items()})

    return {
        'seed': seed,
        'epochs_run': trainer.current_epoch,
        'best_checkpoint': checkpoint.best_model_path,
        'mlflow_run_id': logger.run_id,
        **metrics
    }

In [22]:
results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else []
completed = {entry['seed'] for entry in results}

for seed in SEEDS:
    if seed in completed:
        continue

    entry = train_one(seed)

    if seed == REPORTED_SEED:
        reported_path = MODEL_DIR / f'cnn_seed{REPORTED_SEED}.ckpt'
        shutil.copy(entry['best_checkpoint'], reported_path)
        entry['best_checkpoint'] = str(reported_path)
    else:
        entry['best_checkpoint'] = None
    shutil.rmtree(CHECKPOINT_DIR / f'seed{seed}', ignore_errors=True)

    results.append(entry)
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    print(f"seed {entry['seed']:>3}  epochs {entry['epochs_run']:>3}  "
          f"R2 {entry['r2']:.3f}  RMSE {entry['rmse']:.3f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/home/mykola/.cache/pypoetry/virtualenvs/spectral-core-S9wPqFE3-py3.11/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/mykola/projects/ftir-hba1c-prediction/data/models/_checkpoints/seed0 exists and is not empty.


seed   0  epochs 106  R2 0.734  RMSE 1.214


In [28]:
runs = pd.DataFrame(results).sort_values('seed').reset_index(drop=True)
runs[['seed', 'epochs_run', 'r2', 'r', 'mae', 'mse', 'rmse']]

,seed,epochs_run,r2,r,mae,mse,rmse
0,0,106,0.733514,0.860439,0.909545,1.473231,1.213767
1,1,90,0.733249,0.864745,0.891461,1.474699,1.214372
2,2,106,0.744630,0.865408,0.883935,1.411777,1.188182
3,3,88,0.734330,0.858968,0.909670,1.468719,1.211907
4,4,85,0.740688,0.862673,0.899758,1.433572,1.197319
5,5,99,0.736341,0.859459,0.901011,1.457605,1.207313
6,6,100,0.734401,0.857449,0.907341,1.468328,1.211746
7,7,106,0.734506,0.859231,0.906558,1.467749,1.211507
8,8,105,0.734935,0.858818,0.887800,1.465377,1.210528
9,9,66,0.711118,0.849573,0.929504,1.597043,1.263742


In [24]:
runs[['r2', 'r', 'mae', 'mse', 'rmse']].agg(['mean', 'std', 'min', 'max'])

,r2,r,mae,mse,rmse
mean,0.733771,0.859676,0.902658,1.471810,1.213038
std,0.008746,0.004430,0.013175,0.048352,0.019671
min,0.711118,0.849573,0.883935,1.411777,1.188182
max,0.744630,0.865408,0.929504,1.597043,1.263742


In [27]:
reported_entry = next(entry for entry in results if entry['seed'] == REPORTED_SEED)

print(f"model      {reported_entry['best_checkpoint']}")
print(f"mlflow run {reported_entry['mlflow_run_id']}")
print(f"test R2    {reported_entry['r2']:.3f}")
print(f"mean R2    {runs['r2'].mean():.3f} +/- {runs['r2'].std():.3f} over {len(runs)} seeds")

model      ../data/models/cnn_seed0.ckpt
mlflow run ec51711847c040a2b7306962982df11b
test R2    0.734
mean R2    0.734 +/- 0.009 over 10 seeds


In [ ]:
X_test = np.load(DATA_DIR / 'X_test.npy').astype(np.float32)
y_test = np.load(DATA_DIR / 'y_test.npy').astype(np.float32)

reported = Lightning1DCNNModel.load_from_checkpoint(
    MODEL_DIR / f'cnn_seed{REPORTED_SEED}.ckpt'
)
reported.eval()
with torch.no_grad():
    y_pred = reported(torch.tensor(X_test)).squeeze(-1).numpy()

fig, ax = plot_predictions(y_test, y_pred)
fig.savefig(FIGURE_DIR / 'figure_cnn_test.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURE_DIR / 'figure_cnn_test.pdf', bbox_inches='tight')